# Interconnected PMSM current control

`id_ref, iq_ref → current controller → vd,vq → SVPWM → duties → ideal inverter → Vabc → PMSM`

Physical motor feedback `Ia,Ib,Ic,theta → current feedback transform → id,iq` closes the current loop. Mechanical speed supplies back-EMF and decoupling feedforward.

Each block is a `control.NonlinearIOSystem`, assembled by `build_current_controlled_pmsm()` using explicit named connections. The controller has two continuous PI integral states; the PMSM has four internal states. The motor exposes only physical signals. The assembled drive also exposes controller and feedback diagnostics for plotting.

This is a continuous-time controller and averaged inverter simulation, without a PWM carrier, sample delay, or switching ripple. The output time grid specifies logging times, not controller update times.


In [9]:
from pathlib import Path
import sys
import numpy as np
import control as ct

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "motorsim/models/systems.py").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from motorsim.models.systems import build_current_controlled_pmsm


In [10]:
U_dc = 24 # V
k_rms = 0.03757 # N*m/A_rms
ke_datasheet = 0.02960 # V*s 
R_tt = 0.3 # Ohm
J  = 9.597e-6 # kg*m^2
L_tt = 0.5e-3 # H
L = L_tt / 2 # H
R_s = R_tt / 2 # Ohm
pole_pairs = 4
tau_e = L / R_s # s
P_max = 104 # W
I_max = 6.65 # A (rms??)
M_max = k_rms * I_max # N*m

ke  = ke_datasheet / np.sqrt(3) # V*s/rad
kt  = k_rms * np.sqrt(2)/3

In [11]:
ke, kt

(0.017089567968012923, 0.01771066784611906)

## Parameters and tuning

The electrical time constant is `tau_e = L / R_s`. Choose `bandwidth = 1 / tau_e` (rad/s), with `Kp = L * bandwidth` and `Ki = R_s * bandwidth`.

For this ideal no-load run, external load torque and viscous friction are zero. The motor uses the specified inertia `J`. With no field weakening, the ideal steady-state speed limit is `omega_max = (V_bus / sqrt(3)) / Ke`: at zero current, all available q-axis voltage balances back EMF. This is a model limit, not a mechanical speed rating.

A proportional speed loop requests positive q-axis current, limited to 3 A, and reduces it toward zero as speed approaches this limit. The inner current controller retains its voltage limit and anti-windup. No negative d-axis current is commanded.


In [12]:
motor_params = {
    "Rm": R_s, 
    "Lq": L, 
    "np": pole_pairs,
    "Ke": ke, 
    "Kt": kt,
    "J": J, 
    "b": 0} # 1.30e-4
inverter_params = {"V_bus": U_dc}
bandwidth = 1.0 / tau_e  # rad/s
controller_gains = {
    "Kp": motor_params["Lq"] * bandwidth,
    "Ki": motor_params["Rm"] * bandwidth,
    "Kaw": bandwidth,
}
voltage_limit = inverter_params["V_bus"] / np.sqrt(3)
log_interval = 100e-6
duration = 0.4
omega_max = voltage_limit / motor_params["Ke"]  # mechanical rad/s
speed_gain = 0.1  # A per (rad/s)
iq_limit = 3.0  # peak dq current, A
print(f"Ideal no-load speed: {omega_max:.1f} rad/s ({omega_max * 60 / (2 * np.pi):.0f} rpm)")


Ideal no-load speed: 810.8 rad/s (7743 rpm)


In [13]:
7743  /np.sqrt(2)

5475.127806727437

In [14]:
drive = build_current_controlled_pmsm(
    motor_params, inverter_params, controller_gains
)
print("External inputs:", drive.input_labels)
print("Outputs:", drive.output_labels)
print("States:", drive.state_labels)
# Controller integral_d, integral_q followed by motor iq, id, omega, theta.
x0 = np.zeros(drive.nstates)

def speed_output(t, x, u, params):
    return [np.clip(speed_gain * (omega_max - u[0]), 0.0, iq_limit)]

speed_controller = ct.NonlinearIOSystem(
    None, speed_output, inputs=('omega',), outputs=('iq_ref',), name='speed_controller',
)
no_load_drive = ct.interconnect(
    [speed_controller, drive],
    connections=[
        ['speed_controller.omega', 'current_controlled_pmsm.omega'],
        ['current_controlled_pmsm.iq_ref', 'speed_controller.iq_ref'],
    ],
    inplist=['current_controlled_pmsm.id_ref', 'current_controlled_pmsm.trq'],
    inputs=('id_ref', 'trq'),
    outlist=[*[f'current_controlled_pmsm.{label}' for label in drive.output_labels],
             'speed_controller.iq_ref'],
    outputs=(*drive.output_labels, 'iq_ref'), name='no_load_drive',
)


External inputs: ['id_ref', 'iq_ref', 'trq']
Outputs: ['Ia', 'Ib', 'Ic', 'omega', 'theta', 'i_d', 'i_q', 'vd', 'vq', 'd_a', 'd_b', 'd_c', 'Va', 'Vb', 'Vc']
States: ['current_controller_integral_d', 'current_controller_integral_q', 'dq_pmsm_i_q', 'dq_pmsm_i_d', 'dq_pmsm_omega', 'dq_pmsm_theta']


## Accelerate to no-load maximum speed

Start from rest with zero load torque and zero d-axis current reference. The speed loop supplies up to 3 A during acceleration and tapers the current as back EMF approaches the available voltage. Run long enough to verify speed has settled near the ideal limit.


In [15]:
time = np.linspace(0.0, duration, round(duration / log_interval) + 1)
inputs = np.zeros((2, len(time)))  # id_ref = 0 A, external load torque = 0 Nm
response = ct.input_output_response(
    no_load_drive, time, inputs, X0=x0, return_x=True,
    solve_ivp_kwargs={"rtol": 1e-7, "atol": 1e-9, "max_step": 1e-3},
)
y = response.outputs.T
log = {
    "physical": y[:, :5], "measured": y[:, 5:7],
    "voltage_dq": y[:, 7:9], "duties": y[:, 9:12],
    "v_abc": y[:, 12:15],
    "reference": np.column_stack([np.zeros_like(time), y[:, 15]]),
}
print(f"Simulated {time[-1]:.2f} s with zero external load and zero viscous friction")


Simulated 0.40 s with zero external load and zero viscous friction


In [16]:
assert all(np.isfinite(values).all() for values in log.values())
assert np.max(np.abs(log["physical"][:, :3].sum(axis=1))) < 1e-10
assert log["duties"].min() >= -1e-12 and log["duties"].max() <= 1 + 1e-12
assert np.max(np.linalg.norm(log["voltage_dq"], axis=1)) <= voltage_limit + 1e-10
speed = log["physical"][:, 3]
settled = time > 0.9 * duration
assert np.max(np.abs(speed[settled] - omega_max)) < 0.01 * omega_max, "Speed has not settled within 1% of the ideal limit"
assert np.max(np.abs(log["measured"][settled])) < 0.1, "No-load current has not decayed below 0.1 A"
print(f"Final speed: {speed[-1]:.1f} rad/s ({speed[-1] * 60 / (2 * np.pi):.0f} rpm)")
print(f"Ideal no-load limit: {omega_max:.1f} rad/s ({omega_max * 60 / (2 * np.pi):.0f} rpm)")
print("Final id, iq [A]:", log["measured"][-1])


Final speed: 810.8 rad/s (7743 rpm)
Ideal no-load limit: 810.8 rad/s (7743 rpm)
Final id, iq [A]: [1.73662376e-15 8.57205333e-15]


In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=5, cols=1, shared_xaxes=True,
                    subplot_titles=("Current tracking", "Controller voltages", "Mechanical speed",
                                    "Phase currents", "SVPWM duty cycles"))
for j, axis in enumerate(["d", "q"]):
    fig.add_trace(go.Scatter(x=time, y=log["reference"][:, j],
                            name=f"i{axis} reference", line=dict(dash="dash")), row=1, col=1)
    fig.add_trace(go.Scatter(x=time, y=log["measured"][:, j], name=f"i{axis} feedback"), row=1, col=1)
    fig.add_trace(go.Scatter(x=time, y=log["voltage_dq"][:, j], name=f"v{axis}"), row=2, col=1)
fig.add_trace(go.Scatter(x=time, y=log["physical"][:, 3], name="omega"), row=3, col=1)
for j, phase in enumerate("abc"):
    fig.add_trace(go.Scatter(x=time, y=log["physical"][:, j], name=f"I{phase}"), row=4, col=1)
    fig.add_trace(go.Scatter(x=time, y=log["duties"][:, j], name=f"d{phase}"), row=5, col=1)
for row, units in enumerate(["A", "V", "rad/s", "A", "duty"], start=1):
    fig.update_yaxes(title_text=units, row=row, col=1)
fig.update_xaxes(title_text="Time [s]", row=5, col=1)
fig.add_hline(y=omega_max, line_dash="dash", annotation_text="Ideal no-load limit", row=3, col=1)
fig.update_layout(height=1200, title="PMSM acceleration to no-load maximum speed")
fig.show()
